In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
import datetime
from datetime import datetime, date, timedelta


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration ---
class FakeYFinance:
    def __init__(self):
        self.downloads = []
    def download(self, ticker, start, end):
        self.downloads.append((ticker, start, end))
        offset = 0 if ticker == "AAA" else 10
        return pd.DataFrame({"Close": [100.0 + offset, 101.5 + offset, 103.0 + offset]}, index=pd.date_range(start, periods=3))

FIX_QUANT_TRADING_STRATEGY_BACKTESTER_DATA_LOAD_YFINANCE_DATA_TWO_TICKERS_MIGRATION_ST = SimpleNamespace(write=lambda *a,**k: None, dataframe=lambda df: df, cache_data=lambda f: f)
FIX_QUANT_TRADING_STRATEGY_BACKTESTER_DATA_LOAD_YFINANCE_DATA_TWO_TICKERS_MIGRATION_YF = FakeYFinance()

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration(st, yf):
    @st.cache_data
    def load_yfinance_data_two_tickers(
        ticker1: str, ticker2: str, start_date: datetime.date, end_date: datetime.date
    ) -> pd.DataFrame:
        data1 = yf.download(ticker1, start=start_date, end=end_date)
        data2 = yf.download(ticker2, start=start_date, end=end_date)
        combined_data = pd.DataFrame({"Close_1": data1["Close"], "Close_2": data2["Close"]})
        return combined_data
    return load_yfinance_data_two_tickers

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration(st, yf):
    import datetime



    def load_yfinance_data_two_tickers(
        ticker1: str, ticker2: str, start_date: datetime.date, end_date: datetime.date
    ) -> pl.DataFrame:
        data1 = yf.download(ticker1, start=start_date, end=end_date)
        data2 = yf.download(ticker2, start=start_date, end=end_date)

        df1 = pl.from_pandas(data1.reset_index()).select(
            pl.col(data1.index.name or "Date").alias("date"), pl.col("Close").alias("Close_1")
        )
        df2 = pl.from_pandas(data2.reset_index()).select(
            pl.col(data2.index.name or "Date").alias("date"), pl.col("Close").alias("Close_2")
        )

        combined_data = (
            df1.join(df2, on="date", how="full", coalesce=True)
            .sort("date")
            .select(["Close_1", "Close_2"])
        )
        return combined_data
    return load_yfinance_data_two_tickers

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration ===

# L1 smoke – generated
try:
    _loader = gen_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration(FIX_QUANT_TRADING_STRATEGY_BACKTESTER_DATA_LOAD_YFINANCE_DATA_TWO_TICKERS_MIGRATION_ST, FakeYFinance())
    _r = _loader("AAA", "BBB", date(2020, 1, 1), date(2020, 1, 4))
    print("✅ L1 smoke gen_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _loader = before_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration(FIX_QUANT_TRADING_STRATEGY_BACKTESTER_DATA_LOAD_YFINANCE_DATA_TWO_TICKERS_MIGRATION_ST, FakeYFinance())
    _rb = _loader("AAA", "BBB", date(2020, 1, 1), date(2020, 1, 4))
    print("✅ L1 smoke before_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration(FIX_QUANT_TRADING_STRATEGY_BACKTESTER_DATA_LOAD_YFINANCE_DATA_TWO_TICKERS_MIGRATION_ST, FakeYFinance())("AAA", "BBB", date(2020, 1, 1), date(2020, 1, 4))
    _rg = gen_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration(FIX_QUANT_TRADING_STRATEGY_BACKTESTER_DATA_LOAD_YFINANCE_DATA_TWO_TICKERS_MIGRATION_ST, FakeYFinance())("AAA", "BBB", date(2020, 1, 1), date(2020, 1, 4))
    compare(_rb, _rg, "quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration")
except Exception as _e:
    print(f"❌ L2 equivalence quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration: setup error — {type(_e).__name__}: {_e}")

# L3 edge – same ticker on both sides
try:
    _rb = before_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration(FIX_QUANT_TRADING_STRATEGY_BACKTESTER_DATA_LOAD_YFINANCE_DATA_TWO_TICKERS_MIGRATION_ST, FakeYFinance())("AAA", "AAA", date(2020, 1, 1), date(2020, 1, 4))
    _rg = gen_quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration(FIX_QUANT_TRADING_STRATEGY_BACKTESTER_DATA_LOAD_YFINANCE_DATA_TWO_TICKERS_MIGRATION_ST, FakeYFinance())("AAA", "AAA", date(2020, 1, 1), date(2020, 1, 4))
    compare(_rb, _rg, "L3 edge quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration same ticker")
except Exception as _e:
    print(f"❌ L3 edge quant_trading_strategy_backtester_data_load_yfinance_data_two_tickers_migration: {type(_e).__name__}: {_e}")
